<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde

In [2]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import torch
import pandas as pd

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [4]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [5]:
data = dde.data.TimePDE(
    geomtime,
    pde,
    [],
    num_domain=114,
    train_distribution="uniform"
)

In [6]:
layer_size = [2] + [20] * 2 + [1]
activation = "tanh"
initializer = "Glorot uniform"
net = dde.nn.FNN(layer_size, activation, initializer)

In [7]:
def output_transform(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

In [8]:
net.apply_output_transform(output_transform)

In [9]:
model = dde.Model(data, net)
model.compile("adam", lr=1e-3)
losshistory, train_state = model.train(iterations=1000)

Compiling model...
'compile' took 1.734937 s

Training model...

Step      Train loss    Test loss     Test metric
0         [8.79e+00]    [8.79e+00]    []  
1000      [2.07e-03]    [2.07e-03]    []  

Best model at step 1000:
  train loss: 2.07e-03
  test loss: 2.07e-03
  test metric: []

'train' took 13.528006 s



In [10]:
points = data.train_x
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, np.pi, np.pi, 0, 0], y=[0, 0, 1, 1, 0], mode="lines", fill="toself", fillcolor="rgba(0, 0, 255, 0.05)"))
fig.add_trace(go.Scatter(x=points[:, 0], y=points[:, 1], mode="markers"))
fig.update_layout(xaxis_title="x", yaxis_title="t", showlegend=False)
fig.show()

In [11]:
X, T = np.meshgrid(np.linspace(0, np.pi, 100), np.linspace(0, 1, 100))
inputs = np.column_stack([X.ravel(), T.ravel()])

$$
E_{L_2} = \frac{\| p_{\text{exact}} - p_{\text{pred}} \|_2}{\| p_{\text{exact}} \|_2} \times 100\% = \frac{\sqrt{\sum_{i=1}^{N} (p_{\text{exact}, i} - p_{\text{pred}, i})^2}}{\sqrt{\sum_{i=1}^{N} p_{\text{exact}, i}^2}} \times 100\%
$$

где $N = 10\,000$ — количество узлов равномерной тестовой сетки ($100 \times 100$).


In [12]:
p_pred = model.predict(inputs).ravel()
p_exact = (np.exp(-T) * np.sin(X)).ravel()
l2_error = (np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)) * 100
l2_error

np.float64(1.5427767411327187)

In [13]:
fig = px.line(
    x=X.ravel(),
    y=[model.predict(inputs).ravel(), (np.exp(-T) * np.sin(X)).ravel()],
    animation_frame=np.round(T.ravel(), 3),
    range_y=[-0.05, 1.05],
    labels={'x': 'x', 'value': 'p(x, t)', 'animation_frame': 'Time'}
)
fig.for_each_trace(lambda t: t.update(name='PINN' if t.name == 'wide_variable_0' else 'Exact'))
fig.show()

## Что будет если менять число эпох?

In [14]:
X, T = np.meshgrid(np.linspace(0, np.pi, 100), np.linspace(0, 1, 100))
inputs = np.column_stack([X.ravel(), T.ravel()])

In [15]:
results = {}

for iters in (2000, 3000, 4000):
  models, histories, states = [], [], []
  for i in range(5):
    net = dde.nn.FNN(layer_size, activation, initializer)
    net.apply_output_transform(output_transform)
    model = dde.Model(data, net)
    model.compile("adam", lr=1e-3)
    losshistory, train_state = model.train(iterations=iters)

    models.append(model)
    histories.append(losshistory)
    states.append(train_state)

  results[iters] = {"models": models, "histories": histories, "states": states}

Compiling model...
'compile' took 0.000244 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.49e+01]    [1.49e+01]    []  
1000      [1.97e-03]    [1.97e-03]    []  
2000      [4.35e-04]    [4.35e-04]    []  

Best model at step 2000:
  train loss: 4.35e-04
  test loss: 4.35e-04
  test metric: []

'train' took 7.358081 s

Compiling model...
'compile' took 0.000175 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.32e+00]    [1.32e+00]    []  
1000      [3.75e-04]    [3.75e-04]    []  
2000      [6.37e-05]    [6.37e-05]    []  

Best model at step 2000:
  train loss: 6.37e-05
  test loss: 6.37e-05
  test metric: []

'train' took 8.248375 s

Compiling model...
'compile' took 0.000181 s

Training model...

Step      Train loss    Test loss     Test metric
0         [3.79e+00]    [3.79e+00]    []  
1000      [7.94e-04]    [7.94e-04]    []  
2000      [1.40e-04]    [1.40e-04]    []  

Best model at step 2000:
  train loss

In [17]:
p_exact = (np.exp(-T) * np.sin(X)).ravel()
rows = []

for iters, data_dict in results.items():
  errs = [
      float(
          np.linalg.norm(m.predict(inputs).ravel() - p_exact)
          / np.linalg.norm(p_exact)
          * 100
      )
      for m in data_dict["models"]
  ]
  data_dict["l2_errors"] = errs

  row = {
      "Итерации": iters,
      "Среднее (%)": round(np.mean(errs), 4),
      "Std (%)": round(np.std(errs), 4),
  }
  for idx, e in enumerate(errs, 1):
    row[f"Модель {idx} (%)"] = round(e, 4)

  rows.append(row)

df_results = pd.DataFrame(rows)
display(df_results)

,Итерации,Среднее (%),Std (%),Модель 1 (%),Модель 2 (%),Модель 3 (%),Модель 4 (%),Модель 5 (%)
0,2000,0.4811,0.2248,0.8766,0.4080,0.5003,0.4372,0.1834
1,3000,0.3430,0.1850,0.2296,0.2749,0.2227,0.7102,0.2778
2,4000,0.2945,0.0734,0.2648,0.2855,0.4352,0.2216,0.2655


## Что будет если менять число точек коллокации?

In [18]:
X, T = np.meshgrid(np.linspace(0, np.pi, 100), np.linspace(0, 1, 100))
inputs = np.column_stack([X.ravel(), T.ravel()])

In [19]:
results = {}

for n_domain in (114, 300, 500):
  models, histories, states = [], [], []
  for i in range(5):
    data = dde.data.TimePDE(
        geomtime,
        pde,
        [],
        num_domain=n_domain,
        train_distribution="uniform",
    )
    net = dde.nn.FNN(layer_size, activation, initializer)
    net.apply_output_transform(output_transform)
    model = dde.Model(data, net)
    model.compile("adam", lr=1e-3)
    losshistory, train_state = model.train(iterations=1000)

    models.append(model)
    histories.append(losshistory)
    states.append(train_state)

  results[n_domain] = {
      "models": models,
      "histories": histories,
      "states": states,
  }

Compiling model...
'compile' took 0.000409 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.24e+00]    [2.24e+00]    []  
1000      [1.17e-04]    [1.17e-04]    []  

Best model at step 1000:
  train loss: 1.17e-04
  test loss: 1.17e-04
  test metric: []

'train' took 3.592721 s

Compiling model...
'compile' took 0.000179 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.61e+00]    [1.61e+00]    []  
1000      [5.44e-04]    [5.44e-04]    []  

Best model at step 1000:
  train loss: 5.44e-04
  test loss: 5.44e-04
  test metric: []

'train' took 4.562573 s

Compiling model...
'compile' took 0.000174 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.17e-01]    [1.17e-01]    []  
1000      [6.93e-05]    [6.93e-05]    []  

Best model at step 1000:
  train loss: 6.93e-05
  test loss: 6.93e-05
  test metric: []

'train' took 3.515252 s

Compiling model...
'compile' took 0.000188 s

Traini

In [21]:
import numpy as np
import pandas as pd

p_exact = (np.exp(-T) * np.sin(X)).ravel()
rows = []

for n_domain, data_dict in results.items():
  errs = [
      float(
          np.linalg.norm(m.predict(inputs).ravel() - p_exact)
          / np.linalg.norm(p_exact)
          * 100
      )
      for m in data_dict["models"]
  ]
  data_dict["l2_errors"] = errs

  row = {
      "Точки коллокации": n_domain,
      "Среднее (%)": round(np.mean(errs), 4),
      "Std (%)": round(np.std(errs), 4),
  }
  for idx, e in enumerate(errs, 1):
    row[f"Модель {idx} (%)"] = round(e, 4)

  rows.append(row)

df_results = pd.DataFrame(rows)
display(df_results)

,Точки коллокации,Среднее (%),Std (%),Модель 1 (%),Модель 2 (%),Модель 3 (%),Модель 4 (%),Модель 5 (%)
0,114,0.6758,0.3129,0.3520,0.9519,0.2416,0.9596,0.8740
1,300,0.5684,0.2680,0.2588,1.0434,0.3901,0.6286,0.5210
2,500,0.6435,0.4960,0.8362,0.5517,0.1934,1.4968,0.1396


## Что будет если менять оптимизаторы?

In [22]:
results = {}

data = dde.data.TimePDE(
    geomtime,
    pde,
    [],
    num_domain=114,
    train_distribution="uniform",
)

for opt in ("adam", "rmsprop", "sgd"):
  models, histories, states = [], [], []
  for i in range(5):
    net = dde.nn.FNN(layer_size, activation, initializer)
    net.apply_output_transform(output_transform)
    model = dde.Model(data, net)
    model.compile(opt, lr=1e-3)
    losshistory, train_state = model.train(iterations=1000)

    models.append(model)
    histories.append(losshistory)
    states.append(train_state)

  results[opt] = {
      "models": models,
      "histories": histories,
      "states": states,
  }


Compiling model...
'compile' took 0.000190 s

Training model...

Step      Train loss    Test loss     Test metric
0         [9.88e-01]    [9.88e-01]    []  
1000      [1.96e-04]    [1.96e-04]    []  

Best model at step 1000:
  train loss: 1.96e-04
  test loss: 1.96e-04
  test metric: []

'train' took 3.480011 s

Compiling model...
'compile' took 0.000170 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.25e+00]    [2.25e+00]    []  
1000      [2.30e-04]    [2.30e-04]    []  

Best model at step 1000:
  train loss: 2.30e-04
  test loss: 2.30e-04
  test metric: []

'train' took 3.447963 s

Compiling model...
'compile' took 0.000212 s

Training model...

Step      Train loss    Test loss     Test metric
0         [6.97e-01]    [6.97e-01]    []  
1000      [2.57e-04]    [2.57e-04]    []  

Best model at step 1000:
  train loss: 2.57e-04
  test loss: 2.57e-04
  test metric: []

'train' took 4.493098 s

Compiling model...
'compile' took 0.000194 s

Traini

In [24]:
import numpy as np
import pandas as pd

p_exact = (np.exp(-T) * np.sin(X)).ravel()
rows = []

for opt, data_dict in results.items():
  errs = [
      float(
          np.linalg.norm(m.predict(inputs).ravel() - p_exact)
          / np.linalg.norm(p_exact)
          * 100
      )
      for m in data_dict["models"]
  ]
  data_dict["l2_errors"] = errs

  row = {
      "Оптимизатор": opt,
      "Среднее (%)": round(np.mean(errs), 4),
      "Std (%)": round(np.std(errs), 4),
  }
  for idx, e in enumerate(errs, 1):
    row[f"Модель {idx} (%)"] = round(e, 4)

  rows.append(row)

df_results = pd.DataFrame(rows)
display(df_results)

,Оптимизатор,Среднее (%),Std (%),Модель 1 (%),Модель 2 (%),Модель 3 (%),Модель 4 (%),Модель 5 (%)
0,adam,0.9571,0.3735,0.4678,0.6873,0.8515,1.3830,1.3960
1,rmsprop,5.1335,0.6835,5.8123,5.5170,4.3326,5.7282,4.2775
2,sgd,2.6834,0.7596,1.9547,2.0368,4.0689,2.7692,2.5873
